# 🇳🇬 N-ATLaS Fine-Tuning Quickstart (LoRA on Free Google Colab / GPU)

Welcome! This notebook guides you through adapting **NCAIR1/N-ATLaS** (8B Multilingual African Model by Awarri / NCAIR / NITDA) on your own data using LoRA and evaluating before/after improvements.

### Supported Languages:
- 🇳🇬 **Hausa**
- 🇳🇬 **Igbo**
- 🇳🇬 **Yoruba**
- 🌐 **English**

## 1. Setup Environment & Gated Access
N-ATLaS is a gated model. Request access on [Hugging Face: NCAIR1/N-ATLaS](https://huggingface.co/NCAIR1/N-ATLaS) and provide your token below.

In [ ]:
# Install dependencies (optimized for Google Colab)
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets huggingface_hub pyyaml requests sentencepiece protobuf

import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = input("Enter your Hugging Face Access Token: ")

## 2. Load N-ATLaS with 4-bit Quantization
Fits comfortably in a single 15GB T4 GPU (free Colab tier).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = "NCAIR1/N-ATLaS"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model and LoRA adapters successfully initialized!")

## 3. Prepare Dataset (Aya Multilingual or Custom Domain Data)

In [ ]:
from datasets import load_dataset

# Example: Micro-slice of Aya dataset for Hausa, Igbo, Yoruba
dataset = load_dataset("CohereForAI/aya_dataset", split="train[:500]")
print(f"Loaded {len(dataset)} instruction samples for quick demonstration.")

def format_prompts(batch):
    formatted = []
    for inp, out in zip(batch["inputs"], batch["targets"]):
        messages = [
            {"role": "system", "content": "you are a large language model trained by Awarri AI technologies. You are a friendly assistant and you are here to help."},
            {"role": "user", "content": inp},
            {"role": "assistant", "content": out}
        ]
        formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return {"text": formatted}

train_ds = dataset.map(format_prompts, batched=True)

## 4. Run Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=30,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="outputs",
    ),
)
trainer.train()

## 5. Inference & Verification

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "you are a large language model trained by Awarri AI technologies. You are a friendly assistant and you are here to help."},
    {"role": "user", "content": "Menene amfanin fasaha ga matasan Najeriya?"} # Hausa: What is the benefit of technology to Nigerian youth?
]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.2)
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

## 6. Save & Export (LoRA / GGUF)

In [ ]:
# Save LoRA adapters locally
model.save_pretrained("natlas_lora_adapter")
tokenizer.save_pretrained("natlas_lora_adapter")
print("Adapters saved to 'natlas_lora_adapter'!")

# Optional: Export to 16bit or GGUF
# model.save_pretrained_gguf("natlas_gguf", tokenizer, quantization_method="q4_k_m")